# CW-DETR YOLO BDD100K object-detection training

This Colab notebook trains the CW-DETR Nano detector on a YOLO-format BDD100K export stored in Google Drive.

The provided merged taxonomy has nine object-detection classes. This run is intentionally detection-only: drivable-area masks, lane masks, sign subclassification, learned tracking, and trajectory heads are disabled. Missing YOLO label files are treated as valid background images.

Before running: select `Runtime > Change runtime type > GPU`.

## 1. Mount Google Drive and inspect the GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Enable a GPU runtime before training.')

## 2. Clone CW-DETR

While the focused PR is open, the default ref below points at its branch. Change `REPO_REF` to `main` after the PR is merged.

In [ ]:
# @title Repository settings
REPO_URL = 'https://github.com/pirazor/CW-DETR.git' # @param {type:'string'}
REPO_REF = 'codex/yolo-detection-colab' # @param {type:'string'}
REPO_DIR = '/content/CW-DETR'

import os
import subprocess
from pathlib import Path

if not Path(REPO_DIR).exists():
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', REPO_REF], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'checkout', REPO_REF], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only', 'origin', REPO_REF], check=True)
os.chdir(REPO_DIR)
print('working directory:', os.getcwd())

## 3. Install training dependencies

The backbone contract is pinned to the tested DINOv3 dependency line.

In [ ]:
%pip install -q transformers==4.56.0 timm==1.0.11 einops safetensors scipy pycocotools pyyaml tensorboard tqdm pillow

import transformers, timm
print('transformers:', transformers.__version__)
print('timm:', timm.__version__)

## 4. Authenticate for gated DINOv3 weights

Accept the DINOv3 model terms on Hugging Face first. Set `LOGIN_TO_HUGGING_FACE` to `False` only if this runtime already has an authenticated token.

In [ ]:
# @title Hugging Face login
LOGIN_TO_HUGGING_FACE = True # @param {type:'boolean'}

if LOGIN_TO_HUGGING_FACE:
    from huggingface_hub import notebook_login
    notebook_login()

## 5. Validate the YOLO dataset

The adapter reads `train/images/` and `val/images/` from `data.yaml`, then resolves labels from sibling `train/labels/` and `val/labels/` directories. The segmentation fields in your YAML remain available for other trainers but are not consumed by this object-detection-only run.

The first run writes small `.cwdetr-...-images.txt` manifests beside `data.yaml` in Drive. Later sessions reuse those manifests instead of recursively scanning every image path again. Enable `REBUILD_DATASET_INDEX` only after adding or removing images.

In [ ]:
# @title Dataset and model config
DATA_YAML = '/content/drive/MyDrive/datasets/bdd100k_merged/data.yaml' # @param {type:'string'}
REBUILD_DATASET_INDEX = False # @param {type:'boolean'}
CONFIG = 'configs/cwdetr_nano_yolo_bdd_detection.yaml'

from cwdetr.config import load_config
from cwdetr.data.yolo_detection import YoloDetectionDataset

EXPECTED_NAMES = [
    'car', 'truck', 'bus', 'train', 'bike', 'cyclist', 'person',
    'traffic_light', 'traffic_sign',
]
cfg = load_config(CONFIG)
train_ds = YoloDetectionDataset(
    DATA_YAML, 'train', expected_num_classes=cfg.model.heads.detection.num_classes,
    refresh_index=REBUILD_DATASET_INDEX)
val_ds = YoloDetectionDataset(
    DATA_YAML, 'val', expected_num_classes=cfg.model.heads.detection.num_classes,
    refresh_index=REBUILD_DATASET_INDEX)
assert train_ds.class_names == EXPECTED_NAMES, (train_ds.class_names, EXPECTED_NAMES)
assert not cfg.model.heads.segmentation.enabled
assert not cfg.model.heads.sign_classification.enabled
print('classes:', train_ds.class_names)
print('train images:', len(train_ds))
print('val images:', len(val_ds))
print('train image dir:', train_ds.image_dir)
print('train label dir:', train_ds.label_dir)
print('cached train manifest:', train_ds.index_path)
print('cached val manifest:', val_ds.index_path)

## 6. Visualize one YOLO sample

Adjust `SAMPLE_INDEX` to inspect more labels before starting a long run. Mounted Google Drive occasionally returns a transient `Errno 5` read failure; the loader retries those reads automatically. If Drive remains unavailable after the retries, remount Drive and rerun this cell without rebuilding the dataset index.

In [ ]:
# @title Preview labels
SAMPLE_INDEX = 0 # @param {type:'integer'}

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

sample = train_ds[SAMPLE_INDEX]
image = sample['image']
width, height = image.size
fig, ax = plt.subplots(figsize=(14, 8))
ax.imshow(image)
for label, box in zip(sample['labels'].tolist(), sample['boxes'].tolist()):
    cx, cy, bw, bh = box
    x = (cx - bw / 2) * width
    y = (cy - bh / 2) * height
    rect = Rectangle((x, y), bw * width, bh * height, fill=False, linewidth=2)
    ax.add_patch(rect)
    ax.text(x, y, train_ds.class_names[label], color='white',
            bbox={'facecolor': 'black', 'alpha': 0.6, 'pad': 2})
ax.set_title(sample['image_id'])
ax.axis('off')
plt.show()

## 7. Train the detector

Checkpoints and TensorBoard logs are written to Drive. Start with a small epoch count to validate the pipeline, then increase it for the real run. Leave `RESUME` empty for a fresh run.

In [ ]:
# @title Training settings
OUTPUT_DIR = '/content/drive/MyDrive/cwdetr_runs/bdd100k_yolo_detection' # @param {type:'string'}
EPOCHS = 50 # @param {type:'integer'}
BATCH_SIZE = 4 # @param {type:'integer'}
EVAL_BATCH_SIZE = 4 # @param {type:'integer'}
WORKERS = 0 # @param {type:'integer'}
LEARNING_RATE = 0.0002 # @param {type:'number'}
WARMUP_STEPS = 1000 # @param {type:'integer'}
RESUME = '' # @param {type:'string'}

import shlex
import sys

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
cmd = [
    sys.executable, '-m', 'cwdetr.engine.train',
    '--config', CONFIG,
    '--yolo-data', DATA_YAML,
    '--out', OUTPUT_DIR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--eval-batch-size', str(EVAL_BATCH_SIZE),
    '--workers', str(WORKERS),
    '--lr', str(LEARNING_RATE),
    '--warmup-steps', str(WARMUP_STEPS),
]
if RESUME.strip():
    cmd += ['--resume', RESUME]
print('running:', shlex.join(cmd))
subprocess.run(cmd, check=True)

## 8. Inspect TensorBoard logs

In [ ]:
%load_ext tensorboard
%tensorboard --logdir $OUTPUT_DIR/tensorboard

## 9. Evaluate the best checkpoint

For this detection-only config, `detection/map` and `detection/map50` are the relevant metrics. Segmentation and sign metrics remain zero by design.

In [ ]:
# @title COCO-style validation
BEST_CHECKPOINT = f'{OUTPUT_DIR}/best_detection_map.pth'
eval_cmd = [
    sys.executable, '-m', 'cwdetr.engine.evaluate',
    '--config', CONFIG,
    '--yolo-data', DATA_YAML,
    '--ckpt', BEST_CHECKPOINT,
    '--batch-size', str(EVAL_BATCH_SIZE),
    '--workers', str(WORKERS),
]
print('running:', shlex.join(eval_cmd))
subprocess.run(eval_cmd, check=True)